 **Configure environment**

In [ ]:
from fabrictestbed_extensions.fablib.fablib import FablibManager as fablib_manager
fablib = fablib_manager() 
conf = fablib.show_config()

**Node configuration**

In [ ]:
slice_name="scherrer-original-model" + fablib.get_bastion_username()

node_conf = [
 {'name': "computation_node",   'cores': 16, 'ram': 32, 'disk': 100, 'image': 'default_ubuntu_22', 'packages': []}
]

exp_conf = {'cores': sum([ n['cores'] for n in node_conf]) }

**Reserve resources**

In [ ]:
try:
    slice = fablib.get_slice(slice_name)
    print("You already have a slice by this name!")
    print("If you previously reserved resources, skip to the 'log in to resources' section.")
except:
    print("You don't have a slice named %s yet." % slice_name)
    print("Continue to the next step to make one.")
    slice = fablib.new_slice(name=slice_name)

We will select a random site that has sufficient resources for our experiment:

In [ ]:
while True:
    site_name = fablib.get_random_site()
    if ( (fablib.resources.get_core_available(site_name) > 1.2*exp_conf['cores']) ):
        break

fablib.show_site(site_name)

Then we will add the host

In [ ]:
# this cell sets up the nodes
for n in node_conf:
    slice.add_node(name=n['name'], site=site_name, 
                   cores=n['cores'], 
                   ram=n['ram'], 
                   disk=n['disk'], 
                   image=n['image'])

The following cell submits our request to the FABRIC site. The output of this cell will update automatically as the status of our request changes.

* While it is being prepared, the “State” of the slice will appear as “Configuring”.
* When it is ready, the “State” of the slice will change to “StableOK”.

You may prefer to walk away and come back in a few minutes (for simple slices) or a few tens of minutes (for more complicated slices with many resources).

In [ ]:
slice.submit()


In [ ]:
slice.get_state()
slice.wait_ssh(progress=True)

**Log into resources**

In [ ]:
import pandas as pd
pd.set_option('display.max_colwidth', None)
slice_info = [{'Name': n.get_name(), 'SSH command': n.get_ssh_command()} for n in slice.get_nodes()]
pd.DataFrame(slice_info).set_index('Name')

**Clone original repo and install dependencies**

In [ ]:
# You can also use our modified fluid model in our GitHub repo. We modified it to get the individual flow throughputs and also to visualize the plots with different styles. 
slice.get_node(name="computation_node").execute('git clone https://github.com/simonschdev/imc22-bbr-fluid-model')
slice.get_node(name="computation_node").execute('sudo apt update && sudo apt upgrade -y')
slice.get_node(name="computation_node").execute('sudo apt install -y texlive-latex-base texlive-fonts-recommended texlive-fonts-extra texlive-latex-extra dvipng cm-super')
slice.get_node(name="computation_node").execute('sudo apt install -y python3-pip')
slice.get_node(name="computation_node").execute('cd imc22-bbr-fluid-model; sudo bash ./install_dependencies.sh')

## **Run Model**

### With different config files, you can run different scenarios. We added our modified config files to the github repository.

In [ ]:
# you can run the model for this config file
cmds_run_scenario = '''
            cd /home/ubuntu/imc22-bbr-fluid-model
            ./run_model.py -c configs/model_validation_100Mbps_10ms.yaml
            ./plot.py configs/model_validation_100Mbps_10ms.yaml
            '''

slice.get_node(name="computation_node").execute(cmds_run_scenario)

In [ ]:
# you can download the plots, as an example, you can see how to download fairness plot below.

slice.get_node(name="computation_node").download_file("/home/fabric/work/fairness.pdf",f"/home/ubuntu/imc22-bbr-fluid-model/results/model_validation_100Mbps_10ms/plots/B_fairness_cc_combination_droptail.pdf")                              

### With our updated fluid model code, the generated JSON file after the experiments includes individual flow statistics. We use this information to compute the aggregate throughput of BBR flows. You can also download this file, in addition to the generated PDF plots.